# Pronósticos

En este taller usaremos los modelos de series de tiempo lineales para hacer pronósticos de corto plazo de algunas series económicas

## Series

- PIB frecuencia anual, desde 1980
- PIB frecuencia trimestral, desde marzo 2005
- Tasa de desempleo, frecuencia mensual desde enero 2001
- TRM, frecuencia mensual desde enero 2000

## Procedimiento



In [ ]:
library("readxl")
# Use the RAW url, not the github "blob" page
url <- "https://raw.githubusercontent.com/andvarga-eco/econometrics_finance/main/notebooks/data_forecast.xlsx"

# readxl cannot read straight from a URL, so download to a temp file first
tmp <- tempfile(fileext = ".xlsx")
download.file(url, destfile = tmp, mode = "wb")   # mode = "wb" is required on Windows

# Load the first sheet
pib_y <- read_excel(tmp, sheet = 1)
pib_q<-read_excel(tmp, sheet = 2)
td_m<-read_excel(tmp, sheet = 3)
trm_m<-read_excel(tmp, sheet = 4)

Seguiremos el flujo de trabajo de [Forecasts principles and practice](https://otexts.com/fpp3/arima-r.html). 

In [ ]:
library(fable)
library(tsibble)
library(dplyr)
library(ggtime)
library(feasts)
library(ggplot2)
library(tidyr)

### PIB anual

In [ ]:
pib_y<-pib_y|>filter(date>1979)|>as_tsibble(index=date) # Convertimos en un objeto tsible

In [ ]:
pib_y<-pib_y|>mutate(lpib=log(pib), dlpib=difference(lpib))

In [ ]:
pib_y|>autoplot(lpib)+labs(title="PIB anual, Colombia 1980-2005")

In [ ]:
pib_y|>ACF(lpib)|>autoplot()

La serie en niveles no tiene evidencia de estacionariedad ni dependencia débil. Trabajamos con la serie en diferencias

In [ ]:
pib_y|>gg_tsdisplay(dlpib)

Las gráficas señalan que la serie en diferencias parece ser estacionaria y exhibe dependencia débil. Note que todas las autocorrelaciones son cero, al igual que las parciales, con lo cual no tenemos un criterio evidente para proponer un modelo ARMA en particular. Usaremos el algoritnmo de búsqueda de modelos con el algoritmo de búsqueda de Hyndman-Khandakar

In [ ]:
Fcast_PIB<-pib_y|>
  model(dlpib_step=ARIMA(dlpib),
dlpib_search=ARIMA(dlpib, stepwise=FALSE))

In [ ]:
print(Fcast_PIB)

In [ ]:
tidy(Fcast_PIB)

In [ ]:
glance(Fcast_PIB)

El proceso automático coincide con el diagnóstico visual, pues propone un modelo ARMA(0,0)

Ahora examinamos los residuales

In [ ]:
Fcast_PIB|>select(dlpib_step)|>gg_tsresiduals()

Los residuales son ruido blanco. Ahora hacemos el pronóstico

In [ ]:
Fcast_PIB_values<-Fcast_PIB|>forecast(h=5)|>filter(.model=="dlpib_step")
print(Fcast_PIB_values)

In [ ]:
mean(pib_y$dlpib,na.rm=TRUE)

In [ ]:
Fcast_PIB|>forecast(h=5)|>filter(.model=="dlpib_step")|>autoplot(pib_y)

Note que el pronóstico es la media incondicional de la variable a partir de h=1 ¿Por qué?

## Tasa de desempleo

In [ ]:
td_m<-td_m|>mutate(month=yearmonth(date))|>as_tsibble(index=month) # Convertimos en un objeto tsible

In [ ]:
td_m<-td_m|>mutate(dtd=difference(td_13))

In [ ]:
td_m|>autoplot(td_13)+labs(title="Tasa de desempleo mensual")

In [ ]:
td_m|>ACF(td_13)|>autoplot()+labs(title="ACF TD")

In [ ]:
td_m|>autoplot(dtd)+labs(title="Delta Tasa de desempleo mensual")

In [ ]:
td_m|>ACF(dtd)|>autoplot()+labs(title="ACF DTD")

La serie en niveles no parece estacionaria ni con dependencia débil y exhibe un claro patrón estacional. La serie en diferencias parece estacionaria y muestra dependencia débil. El patrón estacional se evidencia en las autocorrelaciones de cada 12 rezagos. Al modelarla debería incluirse un ajuste por estacionalidad. Usaremos el proceso automatizado de ARIMA sobre la serie en niveles

In [ ]:
Fcast_TD<-td_m|>
  model(td_step=ARIMA(td_13),
td_search=ARIMA(td_13, stepwise=FALSE))

In [ ]:
print(Fcast_TD)

El algoritmo sugiere dos modelos, ambos contienen un componente estacional multiplicativo y sugieren que la serie sea diferenciada. Se diferencian en que td_step no tiene términos AR ni MA, mientras que td_search sugiere un término MA(4)

In [ ]:
tidy(Fcast_TD)

In [ ]:
Fcast_TD_values<-Fcast_TD|>forecast(h=6)
print(Fcast_TD_values)

In [ ]:
Fcast_TD|>forecast(h=12)|>filter(.model=="td_step")|>autoplot(td_m)

In [ ]:
td_m25<-td_m|>filter_index("2025 jan"~ .)
Fcast_TD|>forecast(h=12)|>autoplot(td_m25)

In [ ]:
Fcast_TD|>select(td_search)|>gg_tsresiduals()

### Actividad

- Realice los pronósticos para el PIB trimestral, la TRM, y para una variable de su interés